In [ ]:
import spark_patch
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, avg, length, date_sub
import pandas as pd

spark = SparkSession.builder.appName("InvalidTweets").getOrCreate()

data = [[1, '2015-01-01', 10], [2, '2015-01-02', 25], [3, '2015-01-03', 20], [4, '2015-01-04', 30]]
weather = pd.DataFrame(data, columns=['id', 'recordDate', 'temperature']).astype({'id':'Int64', 'recordDate':'datetime64[ns]', 'temperature':'Int64'})

data = spark.createDataFrame(data=weather)

data.createOrReplaceTempView("weather_view")



data.show()

result_df = spark.sql\
    ("with temp_table as \
(select id, recordDate, temperature, DATE_SUB(recordDate, 1) as prevDate from weather_view)\
Select t.id \
from temp_table t \
inner join weather_view w on t.prevDate = w.recordDate \
where t.temperature > w.temperature")

temp_df = data.select("id", date_sub(col("recordDate"), 1).alias("prevDate"), "temperature")

result_df1 = data.alias("t")\
    .join(temp_df.alias("w"), col("t.recordDate") == col("w.prevDate"))\
    .filter(col("t.temperature") > col("w.temperature"))\
    .select(col("t.id"))

temp_df.show()

result_df1.show()

spark.stop()

+---+-------------------+-----------+
| id|         recordDate|temperature|
+---+-------------------+-----------+
|  1|2015-01-01 00:00:00|         10|
|  2|2015-01-02 00:00:00|         25|
|  3|2015-01-03 00:00:00|         20|
|  4|2015-01-04 00:00:00|         30|
+---+-------------------+-----------+

+---+----------+-----------+
| id|  prevDate|temperature|
+---+----------+-----------+
|  1|2014-12-31|         10|
|  2|2015-01-01|         25|
|  3|2015-01-02|         20|
|  4|2015-01-03|         30|
+---+----------+-----------+

+---+
| id|
+---+
|  2|
+---+

